# EDA 16: Absolute Copy Number (WGS)

## Purpose

`16_OmicsCNGeneWGS.csv` is gene-level copy number, called from **whole-genome sequencing only** —
the narrowest-coverage layer alongside CRISPR (notebook 15). Values are linear-scale relative to
normal diploid: `1.0` = two copies, above = amplification, below = deletion/loss. This notebook
also directly tests `docs/PROJECT_ARCHITECTURE.md` §2's release-mismatch caution by cross-checking
this file's WGS coverage against file 14's WGS-derived `Ploidy`.

**Source, release, grain:** DepMap WGS copy number (augmented delivery, downloaded separately from
`data/raw/`). Row = sequencing run, column = `SYMBOL (EntrezID)`. ~19,955 gene columns.

## Questions this notebook must answer

1. Does the file need default-entry filtering (duplicate sequencing runs per model), like other
   ProfileID/SequencingID-keyed layers?
2. Is missingness scattered or structural — do specific genes have zero coverage everywhere?
3. What does the value distribution look like relative to the diploid baseline of 1.0?
4. Does every model's overall (mean) copy number sit near diploid, or are there outliers to flag?
5. How much of the primary spine has WGS copy-number coverage, and is it lineage-biased?
6. Does this file's WGS coverage agree with file 14's WGS-derived `Ploidy` coverage — a direct test
   of whether both files came from the same release?

**Interpretation rule:** a value is `measured` only for WGS-sequenced lines; every other line is
`not_assayed` for copy number entirely — absence here means "never whole-genome sequenced," not
"no copy-number change.\"

## Column dictionary

| Field | Meaning | Notes |
|---|---|---|
| `SequencingID`, `ModelConditionID`, `ModelID` | identity columns | filter `IsDefaultEntryForModel == 'Yes'` |
| `SYMBOL (EntrezID)` columns | gene | ~19,955 columns |
| cell value | linear copy number relative to diploid | `1.0` = normal, `>1.0` amplified, `<1.0` deleted |

## 1. Basic data contract and default-entry filtering

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 30)
plt.rcParams['figure.dpi'] = 100


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'data' / 'raw' / 'gene_expression').exists():
            return candidate
    raise FileNotFoundError('Run this notebook from inside the az-team25 repo (data/raw/ not found).')


ROOT = find_project_root(Path.cwd())
AUGMENTED = ROOT / 'data' / 'augmented'
NON_GENE_EXPRESSION = ROOT / 'data' / 'raw' / 'non_gene_expression'
NOMENCLATURE = ROOT / 'data' / 'raw' / 'nomenclature'

In [ ]:
%%time
df = pd.read_csv(AUGMENTED / 'gene_properties' / '16_OmicsCNGeneWGS.csv').drop(columns=['Unnamed: 0'], errors='ignore')
meta_cols = ['SequencingID', 'ModelConditionID', 'ModelID', 'IsDefaultEntryForMC', 'IsDefaultEntryForModel']
gene_cols = [c for c in df.columns if c not in meta_cols]
print(f"Raw rows / unique ModelIDs: {len(df)} / {df['ModelID'].nunique()}")
print(f"Gene columns: {len(gene_cols)}")

In [ ]:
df = df[df['IsDefaultEntryForModel'] == 'Yes'].copy()
print(f"Filtered rows / unique ModelIDs: {len(df)} / {df['ModelID'].nunique()}")

### Why this EDA check matters

Same default-profile trap flagged for RNA (notebook 02) and fusions (notebook 05) — repeat
sequencing runs of the same model must be resolved to one canonical row before any per-model
statistic is computed.

## 2. Missingness structure

In [ ]:
gene_data = df[gene_cols]
total_cells = gene_data.shape[0] * gene_data.shape[1]
missing_cells = int(gene_data.isna().sum().sum())
col_miss_pct = gene_data.isna().mean() * 100
fully_missing = int((col_miss_pct == 100).sum())
fully_covered = int((col_miss_pct == 0).sum())

print(f"Total gene x model cells : {total_cells:,}")
print(f"Missing cells            : {missing_cells:,} ({missing_cells/total_cells:.2%})")
print(f"Genes 100% missing       : {fully_missing}")
print(f"Genes 0% missing         : {fully_covered}")
print(f"811-style all-or-nothing check: {fully_missing * gene_data.shape[0] == missing_cells}")

fig, ax = plt.subplots(figsize=(5, 4.5))
ax.bar(['Fully covered\n(0% missing)', 'Fully missing\n(100% missing)'], [fully_covered, fully_missing],
       color=['#4CAF50', '#F44336'])
ax.set_ylabel('Number of genes'); ax.set_title('Gene-column missingness is all-or-nothing')
plt.tight_layout(); plt.show()

### Why this EDA check matters

If missingness is entirely explained by a fixed set of genes with zero coverage across every
model (rather than scattered per-cell gaps), those genes are structurally uncallable by WGS copy
number — typically segmental-duplication/multi-copy gene families the caller routinely excludes —
and should be flagged as `not_assayed` for this layer specifically, not imputed.

## 3. Value distribution and per-model diploid sanity check

In [ ]:
vals = gene_data.values.astype(float).ravel()
vals = vals[~np.isnan(vals)]
print(f"Mean: {vals.mean():.3f}  Median: {np.median(vals):.3f}  Std: {vals.std():.3f}")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.hist(vals[vals < 5], bins=100, color='steelblue', edgecolor='white', alpha=0.85)
ax.axvline(1.0, color='black', linestyle='--', label='Diploid (1.0)')
ax.set_xlabel('Copy number (relative to diploid)'); ax.legend()
ax.set_title('Distribution of copy-number values (clipped at 5 for readability)')
plt.tight_layout(); plt.show()

In [ ]:
model_mean = gene_data.mean(axis=1, skipna=True)
outliers = model_mean[(model_mean < 0.8) | (model_mean > 1.3)]
print(f"Per-model mean CN: min {model_mean.min():.3f}, median {model_mean.median():.3f}, max {model_mean.max():.3f}")
print(f"Models with mean CN outside [0.8, 1.3]: {len(outliers)} of {len(model_mean)}")

### Why this EDA check matters

Confirms the bulk sits near the diploid baseline as expected, and that no model's overall mean CN
is wildly off — a real aneuploid line should show a modest, biologically real deviation, not an
outlier so extreme it signals a data problem instead.

## 4. Coverage of the primary spine

In [ ]:
df9 = pd.read_csv(NOMENCLATURE / '9_DepMap_sample_info.csv', usecols=['DepMap_ID', 'lineage'])
cn_models = set(df['ModelID'].dropna())
spine_models = set(df9['DepMap_ID'].dropna())
matched = cn_models & spine_models
print(f"Primary spine models             : {len(spine_models)}")
print(f"Models with WGS copy number      : {len(matched)} ({len(matched)/len(spine_models):.1%})")
print(f"Models with NO WGS copy number   : {len(spine_models - cn_models)} ({(len(spine_models)-len(matched))/len(spine_models):.1%})")

In [ ]:
covered = df9[df9['DepMap_ID'].isin(matched)]
total_per = df9['lineage'].value_counts()
cov_per = covered['lineage'].value_counts()
pct_cov = (cov_per / total_per * 100).dropna().sort_values()
fig, ax = plt.subplots(figsize=(9, 8))
ax.barh(pct_cov.index, pct_cov.values, color='steelblue')
ax.axvline(pct_cov.mean(), color='red', linestyle='--', label=f'mean={pct_cov.mean():.0f}%')
ax.legend(); ax.set_xlabel('% of lineage with WGS copy-number coverage')
ax.set_title('WGS copy-number coverage by lineage')
plt.tight_layout(); plt.show()

### Why this EDA check matters

Roughly half the catalogue has no WGS copy-number row — absence here is "never whole-genome
sequenced," never "no copy-number change." A researcher's included gene can be genuinely amplified
in an unmeasured line; the correct render is "copy number unknown," not a neutral/absent score.

## 5. Cross-check against file 14's WGS-derived Ploidy — a release-mismatch test

File 14 (`OmicsGlobalSignatures`) carries `Ploidy`, which — like copy number — nominally requires a
WGS run. If both files were generated from the same release, a model's presence here (post-filter)
and a non-null `Ploidy` in file 14 should agree closely. This directly tests
`docs/PROJECT_ARCHITECTURE.md` §2's caution that AZ's delivery mixes DepMap releases.

In [ ]:
df14 = pd.read_csv(NON_GENE_EXPRESSION / '14_OmicsGlobalSignatures.csv',
                    usecols=['ModelID', 'IsDefaultEntryForModel', 'Ploidy'])
df14_filt = df14[df14['IsDefaultEntryForModel'] == 'Yes']
ploidy_models = set(df14_filt.loc[df14_filt['Ploidy'].notna(), 'ModelID'])

both = cn_models & ploidy_models
only_ploidy = ploidy_models - cn_models
only_cn = cn_models - ploidy_models
jaccard = len(both) / len(cn_models | ploidy_models)

print(f"Models with df16 copy number      : {len(cn_models)}")
print(f"Models with df14 Ploidy            : {len(ploidy_models)}")
print(f"In both                            : {len(both)}")
print(f"Only in file 14 (Ploidy, no df16)  : {len(only_ploidy)}")
print(f"Only in df16 (no file 14 Ploidy)   : {len(only_cn)}")
print(f"Jaccard overlap                    : {jaccard:.1%}")

### Why this EDA check matters

Two scores that both nominally require the same WGS run should overlap almost completely if drawn
from the same release. A large, real gap here (not just noise) is concrete evidence of the
release-version mismatch `docs/PROJECT_ARCHITECTURE.md` §2 already warns about — file 14 shipped
with the original `data/raw/` delivery, file 16 was acquired separately into `data/augmented/`.
**Never assume these two files share a model universe** without checking, per that same warning.

## Decisions carried into the next notebooks

1. **Filter to `IsDefaultEntryForModel == 'Yes'` before any per-model use.**
2. Missing genes are structurally missing (all-or-nothing per gene, §2) — flag as
   layer-not_assayed for those specific genes, do not impute.
3. Coverage is ~50% of the spine and lineage-biased (§4) — "copy number unknown" is the correct
   render for the uncovered half, never a neutral score.
4. **File 16 and file 14's `Ploidy` do not fully agree (§5) — do not assume any two
   WGS-derived primary files share a model universe** without checking; this is a live,
   measured instance of the release-mismatch caution in `docs/PROJECT_ARCHITECTURE.md` §2, not a
   hypothetical one.
5. **Limitation:** the illustrative amplification/deletion cutoffs used for description here are
   not the project's actual scoring thresholds — those belong in `scoring/` per root `_.md` §5.